# Assignment

### Leveraging NoSQL functionality in a RDBMS

In our last lecturer we have discovered that we can easily blend json and sql data using PostgreSQL's JSONB datatype and its operators.

Now it's your task to provide some answers using the migrated json data.


## Query jtracks data

We would like to evaluate the data and generate some reports.

### Most popular artists
_Which artists have been listened to most often and how often?_ 

In [1]:
import sqlite3
import pandas as pd

# Connect to the SQLite database file
conn = sqlite3.connect('path_to_your_database_file/music-store_ALLTOGETHER.db')

# Read the data from the database tables into Pandas DataFrames
invoice_lines_df = pd.read_sql_query('SELECT * FROM invoice_lines', conn)
tracks_df = pd.read_sql_query('SELECT * FROM tracks', conn)
albums_df = pd.read_sql_query('SELECT * FROM albums', conn)
artists_df = pd.read_sql_query('SELECT * FROM artists', conn)

# Merge necessary tables to get the artist names and the count of their listened tracks
merged_df = pd.merge(invoice_lines_df, tracks_df, left_on='track_id', right_on='id')
merged_df = pd.merge(merged_df, albums_df, left_on='album_id', right_on='id')
merged_df = pd.merge(merged_df, artists_df, left_on='artist_id', right_on='id')

# Group by artist and count the number of tracks listened to
artist_listen_count = merged_df.groupby('name')['quantity'].sum().reset_index()
artist_listen_count = artist_listen_count.rename(columns={'quantity': 'listen_count'})
artist_listen_count = artist_listen_count.sort_values(by='listen_count', ascending=False)

# Display the result
print(artist_listen_count.head())



Usage:  docker compose [OPTIONS] COMMAND

Define and run multi-container applications with Docker.

Options:
      --ansi string                Control when to print ANSI control
                                   characters ("never"|"always"|"auto")
                                   (default "auto")
      --compatibility              Run compose in backward compatibility mode
      --dry-run                    Execute command in dry run mode
      --env-file stringArray       Specify an alternate environment file.
  -f, --file stringArray           Compose configuration files
      --parallel int               Control max parallelism, -1 for
                                   unlimited (default -1)
      --profile stringArray        Specify a profile to enable
      --progress string            Set type of progress output (auto,
                                   tty, plain, quiet) (default "auto")
      --project-directory string   Specify an alternate working directory
           

unknown docker command: "compose Postgres-Docker.zip"


### Generate an invoice report

The "counts" field represents the number a specific track has been listened to.
Based on the on the pricing information in the tracks table in, generate a report that calculates the total value per track.

One count equals one single order. 


Further Explanation: If the price of a single track is 0.99 and its count equals 10, the total equals 9.90.

### Most valued artists

Similar to the previous question but grouped by the corresponding artist: Using the some information in jtracks, group the total amount by the artist name.

Your result should look like:

| artist_name | total | 
|-------------|-------|
| AC/DC       | 312.23|
| Rammstein   | 200.21|
| Queens      |  33.12| 

### Missing talent

There are some artists that are not in our music-store. Of those missing in our music-store, figure out the ones that should be signed us.

Query the top 5 artists that have the most total count and are not in our music store. Order your result descending.